In [ ]:
# ── Celda 1: Instalación ──────────────────────────────────────────────────────
!pip install -q langchain langchain-mistralai langchain-experimental pandas tabulate

In [8]:
# ── Celda 2: Carga del dataset ────────────────────────────────────────────────
from google.colab import files
import pandas as pd
import io

print("📂 Subí tu archivo 'sales_data_normalizado.csv'")
uploaded = files.upload()

filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))

print(f"✅ Dataset cargado: {df.shape[0]} filas × {df.shape[1]} columnas")
df.head(3)

📂 Subí tu archivo 'sales_data_normalizado.csv'


Saving sales_data_normalizado.csv to sales_data_normalizado.csv
✅ Dataset cargado: 2823 filas × 25 columnas


,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,...,ADDRESSLINE1,ADDRESSLINE2,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE
0,10107,-0.522891,0.596978,2,-0.370825,2003-02-24,Shipped,1,2,2003,...,897 Long Airport Avenue,Desconocido,NYC,NY,10022,USA,Desconocido,Yu,Kwai,Small
1,10121,-0.112201,-0.114450,5,-0.427897,2003-05-07,Shipped,2,5,2003,...,59 rue de l'Abbaye,Desconocido,Reims,Desconocido,51100,France,EMEA,Henriot,Paul,Small
2,10134,0.606505,0.549384,2,0.179443,2003-07-01,Shipped,3,7,2003,...,27 rue du Colonel Pierre Avia,Desconocido,Paris,Desconocido,75508,France,EMEA,Da Cunha,Daniel,Medium


In [9]:
# ── Celda 3: Corpus / contexto del dataset ────────────────────────────────────
import json

# Estadísticas descriptivas de columnas numéricas
numeric_summary = df.describe().round(4).to_string()

# Valores únicos de columnas categóricas clave
categorical_cols = ["STATUS", "PRODUCTLINE", "DEALSIZE", "COUNTRY", "TERRITORY", "YEAR_ID", "QTR_ID"]
cat_summary = {col: df[col].value_counts().to_dict() for col in categorical_cols if col in df.columns}

prefix = f"""
Eres un agente experto en análisis de datos de ventas.
Trabajas con un DataFrame de pandas llamado `df` que contiene datos de órdenes de ventas normalizados.

=== ESTRUCTURA DEL DATASET ===
- Total de filas: {df.shape[0]}
- Total de columnas: {df.shape[1]}
- Columnas: {list(df.columns)}

=== DESCRIPCIÓN DE COLUMNAS ===
- ORDERNUMBER: Identificador único de la orden
- QUANTITYORDERED: Cantidad pedida (normalizada con Z-score)
- PRICEEACH: Precio unitario (normalizado con Z-score)
- ORDERLINENUMBER: Número de línea dentro de la orden
- SALES: Monto de venta (normalizado con Z-score)
- ORDERDATE: Fecha de la orden (formato YYYY-MM-DD)
- STATUS: Estado de la orden — valores: {df['STATUS'].unique().tolist()}
- QTR_ID: Trimestre (1 a 4)
- MONTH_ID: Mes (1 a 12)
- YEAR_ID: Año — valores: {df['YEAR_ID'].unique().tolist()}
- PRODUCTLINE: Línea de producto — valores: {df['PRODUCTLINE'].unique().tolist()}
- MSRP: Precio sugerido de fabricante (normalizado)
- PRODUCTCODE: Código de producto
- CUSTOMERNAME: Nombre del cliente
- PHONE / ADDRESSLINE1 / ADDRESSLINE2 / CITY / STATE / POSTALCODE / COUNTRY: Datos de contacto y ubicación
- TERRITORY: Territorio comercial — valores: {df['TERRITORY'].unique().tolist()}
- CONTACTLASTNAME / CONTACTFIRSTNAME: Contacto del cliente
- DEALSIZE: Tamaño del trato — valores: {df['DEALSIZE'].unique().tolist()}

=== NOTA IMPORTANTE SOBRE NORMALIZACIÓN ===
Las columnas QUANTITYORDERED, PRICEEACH, SALES y MSRP están normalizadas con Z-score.
Valores positivos están por encima del promedio; negativos, por debajo.
Al interpretar resultados, menciona siempre que los valores son Z-scores normalizados.

=== ESTADÍSTICAS DESCRIPTIVAS (numéricas) ===
{numeric_summary}

=== DISTRIBUCIÓN DE VARIABLES CATEGÓRICAS ===
{json.dumps(cat_summary, indent=2, ensure_ascii=False)}

Cuando respondas:
1. Usa siempre el DataFrame `df` para tus cálculos.
2. Responde en español.
3. Aclara cuando los valores son Z-scores normalizados.
4. Sé conciso pero completo en tus explicaciones.
5. Si no podés responder, decilo claramente.
"""

print("✅ Contexto del dataset construido correctamente.")
print(f"   Longitud del corpus: {len(prefix)} caracteres")

✅ Contexto del dataset construido correctamente.
   Longitud del corpus: 4459 caracteres


In [10]:
# ── Celda 4: API Key + LLM (Mistral) + Agente ────────────────────────────────
from google.colab import userdata
from langchain_mistralai import ChatMistralAI
from langchain_experimental.agents import create_pandas_dataframe_agent
import os

# API Key
os.environ["MISTRAL_API_KEY"] = userdata.get("MISTRAL_API_KEY")

# LLM
llm = ChatMistralAI(
    model="mistral-large-latest",
    temperature=0,
)

# Crear el agente
agent = create_pandas_dataframe_agent(
    llm,
    df,
    prefix=prefix,
    verbose=True,
    allow_dangerous_code=True,
    handle_parsing_errors=True,
    agent_type="tool-calling"
)

print("✅ Agente creado correctamente")

/tmp/ipykernel_35539/485524140.py:4: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.agents import create_pandas_dataframe_agent


✅ Agente creado correctamente


/usr/local/lib/python3.12/dist-packages/langchain_experimental/agents/agent_toolkits/pandas/base.py:283: UserWarning: Received additional kwargs {'handle_parsing_errors': True} which are no longer supported.
  warnings.warn(


In [11]:
# ── Celda 5: Interacción con el agente ───────────────────────────────────────
def preguntar(pregunta: str):
    print(f"\n{'='*60}")
    print(f"❓ Pregunta: {pregunta}")
    print('='*60)
    respuesta = agent.invoke({"input": pregunta})
    print(f"\n✅ Respuesta:\n{respuesta['output']}")
    return respuesta['output']

# Ejemplos de uso
preguntar("¿Cuántas órdenes hay por cada línea de producto?")


❓ Pregunta: ¿Cuántas órdenes hay por cada línea de producto?


> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "ordenes_por_linea_producto = df.groupby('PRODUCTLINE')['ORDERNUMBER'].nunique().reset_index()\nordenes_por_linea_producto.columns = ['LÍNEA DE PRODUCTO', 'CANTIDAD DE ÓRDENES']\nordenes_por_linea_producto.sort_values(by='CANTIDAD DE ÓRDENES', ascending=False)"}`


  LÍNEA DE PRODUCTO  CANTIDAD DE ÓRDENES
0      Classic Cars                  199
6      Vintage Cars                  175
5  Trucks and Buses                   73
1       Motorcycles                   72
3             Ships                   65
2            Planes                   59
4            Trains                   45Estas son las cantidades de **órdenes únicas** por cada línea de producto:

| Línea de Producto   | Cantidad de Órdenes |
|---------------------|---------------------|
| Classic Cars        | 199                 |
| Vintage Cars        | 175                 |


'Estas son las cantidades de **órdenes únicas** por cada línea de producto:\n\n| Línea de Producto   | Cantidad de Órdenes |\n|---------------------|---------------------|\n| Classic Cars        | 199                 |\n| Vintage Cars        | 175                 |\n| Trucks and Buses    | 73                  |\n| Motorcycles         | 72                  |\n| Ships               | 65                  |\n| Planes              | 59                  |\n| Trains              | 45                  |'

In [12]:
# ── Celda 6: Loop interactivo ─────────────────────────────────────────────────
print("🤖 Agente de ventas listo. Escribí 'salir' para terminar.\n")

while True:
    pregunta = input("Vos: ")
    if pregunta.strip().lower() in ["salir", "exit", "quit"]:
        print("👋 Sesión finalizada.")
        break
    if pregunta.strip() == "":
        continue
    preguntar(pregunta)

🤖 Agente de ventas listo. Escribí 'salir' para terminar.


❓ Pregunta: ¿Qué clientes son de España?


> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "# Filtrar clientes de España\nespana_clientes = df[df['COUNTRY'] == 'Spain']['CUSTOMERNAME'].unique()\nespana_clientes"}`


['Euro Shopping Channel' 'Corrida Auto Replicas, Ltd' 'Enaco Distributors'
 'CAF Imports' 'Iberia Gift Imports, Corp.']Los clientes que son de **España** en el dataset son los siguientes:

1. Euro Shopping Channel
2. Corrida Auto Replicas, Ltd
3. Enaco Distributors
4. CAF Imports
5. Iberia Gift Imports, Corp.

> Finished chain.

✅ Respuesta:
Los clientes que son de **España** en el dataset son los siguientes:

1. Euro Shopping Channel
2. Corrida Auto Replicas, Ltd
3. Enaco Distributors
4. CAF Imports
5. Iberia Gift Imports, Corp.

❓ Pregunta: ¿Qué clientes son de España?


> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "# Filtrar clientes de Espa